In [7]:
import requests #to get the url from the api
import numpy as np #to build arrays (sets of numbers)
import pandas as pd #dfs
import os as os #pull data from the folder
pd.set_option('display.max_rows', None) # so I can see all rows of a df
pd.set_option('display.max_columns', None)

import os
apikey = os.environ["WMATA_API_KEY"]
stations = pd.read_csv('Data/Output_data/merged_station_data.csv')

In [17]:
def get_path(start_station_code, end_station_code):
    url = f"https://api.wmata.com/Rail.svc/json/jPath?FromStationCode={start_station_code}&ToStationCode={end_station_code}"  #url from wmata
    params =  {"api_key": apikey,
                "Cache-Control": "no-cache" #idk wmata wants it
                }
    response = requests.get(url, params=params) # 
    data = response.json()
    df = pd.DataFrame(data) 
    path = pd.json_normalize(df['Path']) #normalizing data
    return path

In [23]:
stations[stations['Station Name'].isin(['Branch Ave', 'Greenbelt'])]

,Code,Station Name,AvgDaily_Tap_Entries,AvgDaily_NonTapped_Entries,AvgDaily_Entries,NoTap_percent,collectionDays,Name,StationTogether1,StationTogether2,LineCode1,LineCode2,LineCode3,LineCode4,Lat,Lon,Address.Street,Address.City,Address.State,Address.Zip
9,F11,Branch Ave,2323.394587,232.727920,2556.122507,9.104725,1404,Branch Ave,NaN,NaN,GR,NaN,NaN,NaN,38.826995,-76.912134,4707 Old Soper Rd,Suitland,MD,20746
41,E10,Greenbelt,2840.590844,250.308298,3090.899142,8.098236,1398,Greenbelt,NaN,NaN,GR,NaN,NaN,NaN,39.011036,-76.911362,5500 Greenbelt Metro Dr,Beltsville,MD,20705


In [26]:
greenline = get_path('F11', 'E10')
greenline.head(5)

,LineCode,StationCode,StationName,SeqNum,DistanceToPrev
0,GR,F11,Branch Ave,1,0
1,GR,F10,Suitland,2,9144
2,GR,F09,Naylor Road,3,7658
3,GR,F08,Southern Avenue,4,6612
4,GR,F07,Congress Heights,5,5508


In [30]:
greenline_merged = pd.merge(greenline, stations, left_on='StationCode', right_on='Code').drop(columns=['StationCode', 'StationName'])
greenline_merged.head(5)

,LineCode,SeqNum,DistanceToPrev,Code,Station Name,AvgDaily_Tap_Entries,AvgDaily_NonTapped_Entries,AvgDaily_Entries,NoTap_percent,collectionDays,Name,StationTogether1,StationTogether2,LineCode1,LineCode2,LineCode3,LineCode4,Lat,Lon,Address.Street,Address.City,Address.State,Address.Zip
0,GR,1,0,F11,Branch Ave,2323.394587,232.727920,2556.122507,9.104725,1404,Branch Ave,NaN,NaN,GR,NaN,NaN,NaN,38.826995,-76.912134,4707 Old Soper Rd,Suitland,MD,20746
1,GR,2,9144,F10,Suitland,2166.191595,385.247863,2551.439459,15.099236,1404,Suitland,NaN,NaN,GR,NaN,NaN,NaN,38.843891,-76.932022,4500 Silver Hill Rd,Suitland,MD,20746
2,GR,3,7658,F09,Naylor Road,998.625090,217.711111,1216.336201,17.898926,1395,Naylor Road,NaN,NaN,GR,NaN,NaN,NaN,38.851187,-76.956565,3226 Naylor Rd,Hillcrest Heights,MD,20748
3,GR,4,6612,F08,Southern Av,2134.497163,355.436879,2489.934043,14.274952,1410,Southern Avenue,NaN,NaN,GR,NaN,NaN,NaN,38.840974,-76.975360,1411 Southern Ave SE,Hillcrest Heights,MD,20748
4,GR,5,5508,F07,Congress Heights,1526.642553,253.290071,1779.932624,14.230318,1410,Congress Heights,NaN,NaN,GR,NaN,NaN,NaN,38.845334,-76.988170,1290 Alabama Ave SE,Washington,DC,20020


In [33]:
stations[stations['Station Name'].isin(['Shady Grove', 'Glenmont'])]

,Code,Station Name,AvgDaily_Tap_Entries,AvgDaily_NonTapped_Entries,AvgDaily_Entries,NoTap_percent,collectionDays,Name,StationTogether1,StationTogether2,LineCode1,LineCode2,LineCode3,LineCode4,Lat,Lon,Address.Street,Address.City,Address.State,Address.Zip
40,B11,Glenmont,2698.161222,185.605114,2883.766335,6.436205,1408,Glenmont,NaN,NaN,RD,NaN,NaN,NaN,39.061713,-77.053410,12601 Georgia Ave,Silver Spring,MD,20906
74,A15,Shady Grove,5052.685816,294.984397,5347.670213,5.516129,1410,Shady Grove,NaN,NaN,RD,NaN,NaN,NaN,39.119819,-77.164921,215903 Somerville Dr,Rockville,MD,20855


In [40]:
redline = get_path('A15', 'B11')
redline

,LineCode,StationCode,StationName,SeqNum,DistanceToPrev
0,RD,A15,Shady Grove,1,0
1,RD,A14,Rockville,2,14151
2,RD,A13,Twinbrook,3,10586
3,RD,A12,North Bethesda,4,5895
4,RD,A11,Grosvenor-Strathmore,5,7309
5,RD,A10,Medical Center,6,11821
6,RD,A09,Bethesda,7,5530
7,RD,A08,Friendship Heights,8,9095
8,RD,A07,Tenleytown-AU,9,4135
9,RD,A06,Van Ness-UDC,10,5841


In [48]:
fort_totten_code = stations.loc[
    stations['Name'] == 'Fort Totten', 'Code'
].iloc[0]

redline.loc[
    redline['StationName'] == 'Fort Totten', 'StationCode'
] = fort_totten_code

In [50]:
redline_merged = pd.merge(redline, stations, left_on='StationCode', right_on='Code').drop(columns=['StationCode', 'StationName'])
redline_merged.head(5)

,LineCode,SeqNum,DistanceToPrev,Code,Station Name,AvgDaily_Tap_Entries,AvgDaily_NonTapped_Entries,AvgDaily_Entries,NoTap_percent,collectionDays,Name,StationTogether1,StationTogether2,LineCode1,LineCode2,LineCode3,LineCode4,Lat,Lon,Address.Street,Address.City,Address.State,Address.Zip
0,RD,1,0,A15,Shady Grove,5052.685816,294.984397,5347.670213,5.516129,1410,Shady Grove,NaN,NaN,RD,NaN,NaN,NaN,39.119819,-77.164921,215903 Somerville Dr,Rockville,MD,20855
1,RD,2,14151,A14,Rockville,2034.441135,140.390780,2174.831915,6.455247,1410,Rockville,NaN,NaN,RD,NaN,NaN,NaN,39.084215,-77.146424,251 Hungerford Dr,Rockville,MD,20850
2,RD,3,10586,A13,Twinbrook,2008.455256,131.812500,2140.267756,6.158692,1408,Twinbrook,NaN,NaN,RD,NaN,NaN,NaN,39.062359,-77.121113,1600 Chapman Ave,Rockville,MD,20852
3,RD,4,5895,A12,North Bethesda,1908.994326,119.658865,2028.653191,5.898439,1410,North Bethesda,NaN,NaN,RD,NaN,NaN,NaN,39.048043,-77.113131,10000 Citadel Ave,North Bethesda,MD,20852
4,RD,5,7309,A11,Grosvenor-Strathmore,2242.589574,74.796623,2317.386197,3.227629,1362,Grosvenor-Strathmore,NaN,NaN,RD,NaN,NaN,NaN,39.029158,-77.104150,5301 Tuckerman Ln,North Bethesda,MD,20852


In [52]:
# Keep the line label so shared stations can appear on both routes.
metroLines = pd.concat([
    greenline_merged,
    redline_merged
], ignore_index=True)
metroLines.to_csv('Data/Output_data/metroLines_merged_station_data.csv', index=False)
metroLines.head(5)

,LineCode,SeqNum,DistanceToPrev,Code,Station Name,AvgDaily_Tap_Entries,AvgDaily_NonTapped_Entries,AvgDaily_Entries,NoTap_percent,collectionDays,Name,StationTogether1,StationTogether2,LineCode1,LineCode2,LineCode3,LineCode4,Lat,Lon,Address.Street,Address.City,Address.State,Address.Zip
0,GR,1,0,F11,Branch Ave,2323.394587,232.727920,2556.122507,9.104725,1404,Branch Ave,NaN,NaN,GR,NaN,NaN,NaN,38.826995,-76.912134,4707 Old Soper Rd,Suitland,MD,20746
1,GR,2,9144,F10,Suitland,2166.191595,385.247863,2551.439459,15.099236,1404,Suitland,NaN,NaN,GR,NaN,NaN,NaN,38.843891,-76.932022,4500 Silver Hill Rd,Suitland,MD,20746
2,GR,3,7658,F09,Naylor Road,998.625090,217.711111,1216.336201,17.898926,1395,Naylor Road,NaN,NaN,GR,NaN,NaN,NaN,38.851187,-76.956565,3226 Naylor Rd,Hillcrest Heights,MD,20748
3,GR,4,6612,F08,Southern Av,2134.497163,355.436879,2489.934043,14.274952,1410,Southern Avenue,NaN,NaN,GR,NaN,NaN,NaN,38.840974,-76.975360,1411 Southern Ave SE,Hillcrest Heights,MD,20748
4,GR,5,5508,F07,Congress Heights,1526.642553,253.290071,1779.932624,14.230318,1410,Congress Heights,NaN,NaN,GR,NaN,NaN,NaN,38.845334,-76.988170,1290 Alabama Ave SE,Washington,DC,20020
